# ReliableML overview: a statistical workflow for machine-learning reliability

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/overview.ipynb)

Companion notebook for the ReliableML paper `docs/latex/overview.tex`. It walks through the three stages of the paper - data partitioning, meta-learning, and post-deployment evidence - with one small end-to-end run.

All numbers shown below are computed by the cells themselves (fixed random seeds); nothing is hard-coded. Run the cells top to bottom.

## 0. Setup (parameters)

**Parameter cell - choose ONE installation source.** The default installs the released package from PyPI. To try the unreleased development version, comment out option A and uncomment option B. If neither works (for example a restricted network), run the fallback cell below it and upload a zip of the repository.

In [ ]:
#@title PARAMETERS: install ReliableML
# ---- Option A (default): released version from PyPI ----
!pip install -q ReliableML
# ---- Option B: unreleased version from GitHub (uncomment; comment option A) ----
# !pip install -q "git+https://github.com/Jabualani/ML-Reliabilty.git"
# Plots need matplotlib (preinstalled on Colab); locally: pip install matplotlib
# Optional: ATTP (S-ATTP partitioning, used in stage 1 when available)
!pip install -q i-attp==0.2.3

In [ ]:
#@title Fallback: upload a repository zip (only runs if ml_reliability is not importable)
import importlib.util, pathlib, sys, zipfile

if importlib.util.find_spec("ml_reliability") is None:
    from google.colab import files  # Colab only

    # GitHub: Code > Download ZIP, then pick the file in the dialog
    uploaded = files.upload()
    for archive in uploaded:
        zipfile.ZipFile(archive).extractall("repo_zip")
    package_init = next(pathlib.Path("repo_zip").rglob("ml_reliability/__init__.py"))
    sys.path.insert(0, str(package_init.parent.parent))
    # the repo's dependencies (numpy, pandas, scikit-learn, scipy, statsmodels) are
    # preinstalled on Colab; otherwise: !pip install -q numpy pandas scikit-learn scipy statsmodels

import ml_reliability
print("ml_reliability loaded from:", pathlib.Path(ml_reliability.__file__).parent)

## 1. The workflow and its key statistics

Let $\mathcal D=\{(x_i,y_i)\}_{i=1}^n$ be labeled development data and $P_{\mathrm{ref}}$ the feature distribution represented at fit time. The paper separates **what needs labels** from **what does not**:

| Evidence | Needs | API |
|---|---|---|
| Feature shift (PSI, KL, KS, ...) | unlabeled batch + fitted reference | `predict_with_report`, `assess_reliability` |
| Predictive evaluation, calibration | labeled representative batch | `evaluate` |
| Group rates (fairness) | labels + protected groups + positive class | `evaluate(sensitive_features=...)` |
| Deterioration | $\ge 3$ dated labeled batches | `assess_deterioration` |
| Explanation | background data + target class | `explain` |
| Reason audit | labeled holdout + background | `audit_reasons` |

**Cross-fitted meta-learning.** With $K$ stratified folds and $M$ base estimators, row $i$ receives out-of-fold class probabilities $\hat p_{im}^{(-k(i))}$ and
$$Z_i=\big[\hat p_{i1,1}^{(-k(i))},\ldots,\hat p_{i1,C}^{(-k(i))},\ldots,\hat p_{iM,1}^{(-k(i))},\ldots,\hat p_{iM,C}^{(-k(i))}\big],\qquad \hat p(y\mid x)=g\big(Z(x)\big),$$
where $g$ is fitted to $(Z_i,y_i)$ and, at inference, base models refitted on all training rows produce $Z(x)$.

**Reference-based drift.** With reference proportions $p_b$ and current proportions $q_b$ (smoothed by $\epsilon>0$),
$$\operatorname{PSI}(p,q)=\sum_b (q_b-p_b)\log\frac{q_b}{p_b},\qquad D_{\mathrm{KL}}(p\Vert q)=\sum_b p_b\log\frac{p_b}{q_b},\qquad S_{\mathrm{display}}=\exp(-\overline{\operatorname{PSI}}).$$
$S_{\mathrm{display}}$ is a descriptive transform, **not** a probability of correctness.

**Multiclass MCC** from a confusion matrix with $t_k$ true counts, $p_k$ predicted counts, $c=\sum_k C_{kk}$, $s=\sum_{ij}C_{ij}$:
$$\operatorname{MCC}=\frac{cs-\sum_k t_kp_k}{\sqrt{(s^2-\sum_k p_k^2)(s^2-\sum_k t_k^2)}}.$$

**Group gaps.** For a rate $r_a$ (selection rate, TPR, FPR, PPV) over groups $a\in\mathcal A$: $\Delta_r=\max_a r_a-\min_a r_a$.

**Deterioration.** For dated batches with accuracy $a_j$ at time $t_j$: $\hat\beta_1=\frac{\sum_j (t_j-\bar t)(a_j-\bar a)}{\sum_j (t_j-\bar t)^2}$; deterioration is flagged only if the bootstrap interval for $\hat\beta_1$ lies wholly below $0$.

**Shapley attribution.** $\phi_j(f,x)=\sum_{S\subseteq F\setminus\{j\}}\frac{|S|!\,(d-|S|-1)!}{d!}\,[v_x(S\cup\{j\})-v_x(S)]$, estimated by Monte Carlo permutations.

## 2. Data and stage 1: keep a final evaluation set, then partition

We use scikit-learn's breast-cancer dataset (a bundled data source) with six numeric features. A final evaluation set is held out *first*; the (optional) ATTP partition is built from development data only, because S-ATTP chooses its split using its own test score and is therefore not an unbiased final estimate.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from ml_reliability import (
    MetaPipelineClassifier,
    ReliablePipeline,
    assess_deterioration,
)

SEED = 42
FEATURES = ["mean radius", "mean texture", "mean smoothness",
            "mean compactness", "mean symmetry", "mean fractal dimension"]
data = load_breast_cancer(as_frame=True)
X_all, y_all = data.data[FEATURES], data.target
print("rows:", len(X_all), "| class counts:", y_all.value_counts().to_dict())

# Untouched final evaluation set, separated before any partitioning or model selection.
X_dev, X_final, y_dev, y_final = train_test_split(
    X_all, y_all, test_size=0.3, stratify=y_all, random_state=SEED)

try:
    import attp  # optional companion package (pip install i-attp==0.2.3)
    part = attp.s_attp_clf(X_dev, y_dev, random_state=SEED)
    X_train, y_train, X_part_test, y_part_test = part
    print(f"S-ATTP: split={part.best_split}, clusters={part.best_n_clusters}, "
          f"LDA comps={part.n_lda_components}, internal test score={part.test_score:.3f} "
          "(a tuning criterion, not a final estimate)")
except ImportError:
    X_train, X_part_test, y_train, y_part_test = train_test_split(
        X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED)
    print("ATTP not installed -> plain stratified split of the development data")
print("train rows:", len(X_train), "| partition-test rows:", len(X_part_test), "| final rows:", len(X_final))

## 3. Stage 2: fit a cross-fitted meta-learner inside `ReliablePipeline`

`ReliablePipeline.fit` fits a `DriftMonitor` on the training features (the reference), then fits the estimator, and saves a reference-distribution plot.

In [ ]:
base = [
    ("logreg", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
    ("tree", DecisionTreeClassifier(max_depth=3, random_state=SEED)),
]
model = ReliablePipeline(
    MetaPipelineClassifier(base_estimators=base, n_folds=3, random_state=SEED),
    drift_threshold=0.2,
    plot_dir="reliability_plots",
    bias_variance_folds=3,
    bias_variance_repeats=2,
).fit(X_train, y_train)

print("meta-classifier:", type(model.estimator.meta_classifier_).__name__)
fig, _ = model.plot_reference_distributions()
plt.show()

## 4. Stage 3a: unlabeled monitoring at prediction time

A copy of the final set is shifted (`mean radius` scaled by 1.3, plus noise) to mimic a population change. No labels are used here.

In [ ]:
rng = np.random.default_rng(SEED)
X_shifted = X_final.copy()
X_shifted["mean radius"] = X_shifted["mean radius"] * 1.3 + rng.normal(0, 0.5, len(X_shifted))

rows = []
for name, batch in [("final (same population)", X_final), ("shifted copy", X_shifted)]:
    _, report = model.predict_with_report(batch)
    rows.append({
        "batch": name,
        "similarity exp(-mean PSI)": round(report.similarity_score, 3),
        "mean PSI": round(report.mean_psi, 3),
        "covariate shift flagged": report.covariate_shift_detected,
        "drifted features": ", ".join(report.drifted_features) or "-",
    })
pd.DataFrame(rows)

In [ ]:
fig, _ = model.plot_distributions(X_shifted, feature_names=["mean radius", "mean texture"])
plt.show()

## 5. Stage 3b: labeled analyses once outcomes arrive

`evaluate` adds classification metrics (including MCC), probability calibration, statistical tests and a repeated-CV Brier decomposition. The protected attribute below is **synthetic** (a texture split, passed separately from the model features) purely to show the report layout.

In [ ]:
group = np.where(X_final["mean texture"] > X_final["mean texture"].median(), "high_texture", "low_texture")
evaluation = model.evaluate(X_final, y_final, sensitive_features={"texture_group": group}, positive_label=1)

clf = evaluation.classification
print("accuracy:", round(clf["accuracy"], 3), "| balanced accuracy:", round(clf["balanced_accuracy"], 3),
      "| MCC:", round(clf["matthews_correlation_coefficient"], 3))
print("confusion matrix (rows=true, cols=pred):", clf["labels"], clf["confusion_matrix"])
print("Brier:", round(clf["probability_calibration"]["mean_classwise_brier_score"], 4),
      "| ECE:", round(clf["probability_calibration"]["top_label_expected_calibration_error"], 4))
bv = evaluation.bias_variance
print("empirical Brier decomposition: bias^2 =", round(bv["bias_squared"], 4),
      "| variance =", round(bv["prediction_variance"], 4))

fair = evaluation.fairness["texture_group"]["classes"]["1"]
pd.DataFrame(fair["per_group"]).round(3)

In [ ]:
print("statistical-parity gap:", round(fair["statistical_parity"]["gap"], 3),
      "CI", [round(v, 3) for v in fair["statistical_parity"]["confidence_interval"]])
print("equalized-odds max gap:", round(fair["equalized_odds"]["maximum_gap"], 3))
print("predictive-parity gap :", round(fair["predictive_parity"]["gap"], 3))

### Deterioration over dated batches (simulated drift)

Six monthly batches are resampled from the final set with a growing shift in `mean radius`; accuracy of the fitted model on each batch is regressed on time.

In [ ]:
history = []
dates = pd.date_range("2026-01-01", periods=6, freq="MS")
for step, date in enumerate(dates):
    idx = rng.choice(len(X_final), size=120, replace=True)
    batch = X_final.iloc[idx].copy()
    batch["mean radius"] = batch["mean radius"] * (1 + 0.12 * step)
    history.append({"timestamp": date.date().isoformat(),
                    "y_true": y_final.iloc[idx].to_numpy(),
                    "y_pred": model.estimator.predict(batch)})
det = assess_deterioration(history, bootstrap_samples=500)
print("accuracy per batch:", [round(float(o["accuracy"]), 3) for o in det.observations])
print("slope (accuracy/day):", f"{det.accuracy_slope:.5f}",
      "| 95% CI:", [f"{v:.5f}" for v in det.slope_confidence_interval],
      "| deterioration flagged:", det.deterioration_detected)

### Explanation and reason audit (brief)

Both need a probability-capable classifier and finite numeric features. Sample counts are kept small here for speed; increase `num_samples`/`explanation_samples` for real use.

In [ ]:
explanation = model.explain(X_final, class_label=1, max_instances=15, num_samples=100)
print(pd.DataFrame(explanation.global_shap_importance).round(4).to_string(index=False))
al = explanation.reason_alignment
print("mean Spearman agreement SHAP vs LIME:", round(al["mean_absolute_importance_spearman"], 3),
      "| 95% CI", [round(v, 3) for v in al["mean_spearman_confidence_interval"]])

audit = model.audit_reasons(X_final, y_final, positive_label=1, bootstrap_samples=100,
                            explanation_samples=100, max_explained_instances=15)
pv = audit.predictive_validity
print("audit accuracy:", round(pv["accuracy"], 3), "CI", [round(v, 3) for v in pv["accuracy_confidence_interval"]],
      "| Brier:", round(pv["brier_score"], 3), "| ECE:", round(pv["top_label_expected_calibration_error"], 3))
pd.DataFrame(audit.feature_reliance)[["feature", "loss_increase", "loss_increase_confidence_interval", "positive_lower_bound"]]

## Interpretation and limitations

**How to read the output.** Drift output is about the *inputs* only: a high similarity or a missing flag says nothing about accuracy. Accuracy, MCC, calibration, group gaps, deterioration slope, explanations and audit reliance each answer a different question and need the data listed in the table in section 1. Intervals are bootstrap intervals that assume independent rows.

**Limitations.** Explanation agreement is not correctness or causality; fairness gaps are descriptive and the group here is synthetic; the deterioration run is a simulation with a linear trend; the S-ATTP internal score is a tuning criterion. Pickled artifacts can execute code when loaded - load only trusted files. No integrated benchmark or superiority claim is made.

These notebooks are methodological demonstrations on synthetic data and small bundled scikit-learn datasets. They do not show that any method improves performance on your data; that requires a prespecified evaluation on independent data.